In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
ROOT_DIR = Path('/projects/ai_science_alphafold')
RAW_DIR = ROOT_DIR / 'raw'
OUT_DIR = ROOT_DIR / 'processed'

run_config = pd.read_json(RAW_DIR / 'run_config.json')
USE_OPENALEX = bool(run_config.loc[0, 'USE_OPENALEX'])
print(run_config.to_dict(orient='records')[0])

In [ ]:
pretrend_pub2year = pd.read_csv(OUT_DIR / 'pretrend_pub2year.csv')
pub2fos = pd.read_csv(OUT_DIR / 'pub2fos.csv')
pub2aff_main = pd.read_csv(OUT_DIR / 'pub2aff_main.csv')
pub2ids = pd.read_csv(OUT_DIR / 'pub2ids.csv')
aff2rank = pd.read_csv(OUT_DIR / 'aff2rank.csv')

pdb2citation = pd.read_csv(RAW_DIR / 'pdb.csv', usecols=['structureId', 'pubmedId'])
pdb2citation = pdb2citation.loc[pdb2citation['pubmedId'].astype(str).ne('')].copy()
pdb2citation['pubmedId'] = pd.to_numeric(pdb2citation['pubmedId'], errors='coerce').astype('Int64')
pdb2citation = pdb2citation.loc[pdb2citation['pubmedId'].notna()].copy()

focal_pubs = pdb2citation[['pubmedId']].drop_duplicates()
focal_pubs['pubmedId'] = focal_pubs['pubmedId'].astype('string')
focal_pubs = focal_pubs.merge(pub2ids, left_on='pubmedId', right_on='pmid', how='inner')

if USE_OPENALEX:
    focal_pubs['oaid_int'] = focal_pubs['oaid'].astype(str).str.lstrip('W').astype('int64')
    focal_pubs = focal_pubs.merge(pretrend_pub2year, left_on='oaid_int', right_on='PaperID', how='inner')
else:
    focal_pubs = focal_pubs.merge(pretrend_pub2year, left_on='magid', right_on='PaperID', how='inner')

print('# Papers indexed in PDB:', focal_pubs.shape[0])

In [ ]:
if USE_OPENALEX:
    pub2aff_main = pub2aff_main.copy()
    pub2aff_main['oaid_int'] = pub2aff_main['oaid'].astype(str).str.lstrip('W').astype('int64')

treatment_scientists = (
    focal_pubs[['oaid']]
    .drop_duplicates()
    .merge(
        pub2aff_main,
        on='oaid',
        how='inner',
    )
)

print('# Treated Scientists:', treatment_scientists['author_id'].nunique())

In [ ]:
treated_paper_ids = set(treatment_scientists['oaid'].astype(str).str.lstrip('W').astype('int64'))

candidate_papers = pretrend_pub2year.copy()
candidate_papers['is_pdb_paper'] = candidate_papers['PaperID'].isin(treated_paper_ids)
journal_year_has_treated = (
    candidate_papers.groupby(['JournalID', 'Year'], dropna=False)['is_pdb_paper'].transform('sum') > 0
)
candidate_papers = candidate_papers.loc[journal_year_has_treated].copy()

fos_by_paper = pub2fos.groupby('PaperID', as_index=False)['FieldID'].agg(list)
candidate_papers['bio_med_fos'] = candidate_papers['PaperID'].isin(set(pub2fos['PaperID']))
candidate_papers = candidate_papers.merge(fos_by_paper, on='PaperID', how='left')

print('# Structural Biology Journals:', candidate_papers['JournalID'].nunique())
print('# related BioMed papers:', candidate_papers.shape[0])

In [ ]:
pub2aff_main_join = pub2aff_main.copy()
pub2aff_main_join['magid'] = pub2aff_main_join['oaid'].astype(str).str.lstrip('W').astype('int64')

candidate_scientists = candidate_papers.merge(
    pub2aff_main_join,
    left_on='PaperID',
    right_on='magid',
    how='inner',
)

candidate_scientists['from_bio_med'] = candidate_scientists.groupby('author_id', dropna=False)['bio_med_fos'].transform(
    'mean'
)
candidate_scientists = candidate_scientists.loc[candidate_scientists['from_bio_med'] > 0].copy()

author_treated_count = candidate_scientists.groupby('author_id', dropna=False)['is_pdb_paper'].transform('sum')
print('# Treated Authors:', candidate_scientists.loc[author_treated_count > 0, 'author_id'].nunique())
print('# Control Authors:', candidate_scientists.loc[author_treated_count == 0, 'author_id'].nunique())

In [ ]:
top_aff_set = set(aff2rank['institution_id'].dropna().astype(str))
candidate_scientists['is_top_aff'] = candidate_scientists['affs_id'].map(
    lambda xs: any((x in top_aff_set) for x in (xs if isinstance(xs, list) else []))
)

candidate_scientists.to_csv(OUT_DIR / 'candidate_scientists.csv', index=False)
candidate_papers.to_csv(OUT_DIR / 'candidate_papers.csv', index=False)
treatment_scientists.to_csv(OUT_DIR / 'treatment_scientists.csv', index=False)

print('Saved cohort tables to', OUT_DIR)